# Capítulo 20 — Estandarizar nombres y crear nuevas columnas

## Breve repaso

En los capítulos anteriores avanzamos sobre varias tareas centrales de preparación de datos. Trabajamos con valores faltantes, analizamos distintas estrategias para tratarlos, revisamos duplicados, limpiamos textos y categorías, convertimos columnas al tipo de dato adecuado y profundizamos en el trabajo con fechas.

Ese recorrido nos permitió ver una idea importante: preparar un dataset no consiste en aplicar una sola transformación, sino en tomar varias decisiones conectadas entre sí. Cada paso mejora algún aspecto de la tabla y deja los datos en mejores condiciones para el análisis.

En este capítulo vamos a trabajar con dos tareas que suelen aparecer en una etapa avanzada de preparación: estandarizar nombres de columnas y crear nuevas columnas útiles para el análisis.

Ya trabajamos antes con nombres de columnas, así que no vamos a tratar este tema como algo completamente nuevo. Esta vez lo retomaremos dentro de un flujo de limpieza más realista. La idea será justificar por qué conviene usar nombres consistentes y aplicar una estrategia simple para transformar columnas como `Transaction ID`, `Price Per Unit` o `Transaction Date` en nombres más cómodos para trabajar.

Luego vamos a crear nuevas columnas a partir de datos ya existentes. Por ejemplo, si tenemos una columna de cantidad y otra de precio unitario, podemos calcular un importe total. También podemos crear columnas de validación para comparar un valor registrado con un valor calculado, o columnas derivadas de una fecha.

Estas nuevas columnas no son adornos. Muchas veces permiten responder preguntas que el dataset original no podía responder directamente, o ayudan a verificar la consistencia interna de los datos.

Al finalizar este capítulo deberías poder:

- Comprender por qué conviene estandarizar nombres de columnas.
- Renombrar columnas usando un criterio uniforme.
- Verificar que los nuevos nombres quedaron correctamente aplicados.
- Crear columnas nuevas a partir de operaciones entre columnas existentes.
- Crear columnas de validación para comparar valores registrados y calculados.
- Crear columnas derivadas de una fecha.
- Comprender que crear columnas nuevas puede enriquecer el análisis y también ayudar a controlar la calidad de los datos.

## Dataset de trabajo

Para este capítulo vamos a volver al dataset real **Cafe Sales — Dirty Data for Cleaning Training**.

Este dataset nos permite trabajar con varias situaciones ya vistas: columnas con nombres que contienen espacios y mayúsculas, columnas numéricas que necesitan conversión, una columna de fecha y una relación interna entre cantidad, precio unitario e importe total.

Como venimos haciendo en esta parte del recorrido, vamos a cargar el dataset desde Kaggle. En Google Colab, `kagglehub` ya suele estar disponible, por lo que comenzamos directamente con las importaciones necesarias.

In [17]:
import kagglehub
import pandas as pd
import os

ruta_dataset = kagglehub.dataset_download(
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training"
)

archivos = os.listdir(ruta_dataset)

archivos_csv = [
    archivo for archivo in archivos
    if archivo.endswith(".csv")
]

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])

df = pd.read_csv(ruta_csv)

df.head()

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


La salida de `head()` nos permite confirmar que el archivo fue cargado correctamente.

En este capítulo vamos a trabajar sobre una copia del dataset. De esa manera podemos transformar nombres, convertir columnas y crear nuevas variables sin perder el punto de partida original.

## Por qué estandarizar nombres de columnas

Los nombres de columnas son una parte importante de la estructura del dataset.

En el archivo original encontramos nombres como:

```text
Transaction ID
Price Per Unit
Total Spent
Payment Method
Transaction Date
```

Estos nombres son comprensibles, pero no siempre son los más cómodos para trabajar en código. Tienen espacios, usan mayúsculas y están escritos con un estilo más cercano a un encabezado de planilla que a un nombre de variable.

En Pandas podemos trabajar con nombres así sin problema, usando corchetes:

```python
df["Price Per Unit"]
```

Sin embargo, en un flujo de análisis largo puede ser más cómodo usar nombres consistentes, en minúsculas y con guiones bajos:

```python
df["price_per_unit"]
```

Estandarizar nombres no cambia los valores del dataset. Lo que cambia es la forma en que nos referimos a las columnas. Esto puede reducir errores de escritura, hacer que el código sea más legible y facilitar la aplicación de transformaciones encadenadas.

En esta etapa vamos a adoptar un criterio simple:

```text
usar minúsculas
reemplazar espacios por guiones bajos
mantener nombres descriptivos
```

Ese criterio suele conocerse como estilo `snake_case`, porque las palabras se escriben en minúsculas y separadas por guiones bajos.

In [18]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

La salida muestra los nombres originales de las columnas.

Antes de crear nuevas columnas o aplicar transformaciones más largas, vamos a construir una copia del dataset y renombrar sus columnas con un criterio más uniforme.

## Renombrar columnas con un criterio uniforme

Para estandarizar los nombres de columnas podemos usar `rename()`.

En este caso vamos a crear una copia llamada `df_trabajo` y vamos a renombrar las columnas principales usando nombres en minúsculas y con guiones bajos.

No estamos cambiando los datos. Solo estamos cambiando los nombres con los que vamos a referirnos a cada columna.

In [19]:
df_trabajo = df.copy()

df_trabajo = df_trabajo.rename(columns={
    "Transaction ID": "transaction_id",
    "Item": "item",
    "Quantity": "quantity",
    "Price Per Unit": "price_per_unit",
    "Total Spent": "total_spent",
    "Payment Method": "payment_method",
    "Location": "location",
    "Transaction Date": "transaction_date"
})

df_trabajo.head()

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


Ahora el `DataFrame` tiene nombres de columnas más cómodos para trabajar.

Podemos verificar la estructura revisando nuevamente los nombres:

In [20]:
df_trabajo.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

Los nombres quedaron en un formato más uniforme.

Este paso parece pequeño, pero puede facilitar mucho el trabajo posterior. Por ejemplo, al escribir transformaciones más largas, nombres como `price_per_unit` o `transaction_date` suelen ser más cómodos y menos propensos a errores que nombres con espacios y mayúsculas.

También prepara el terreno para crear nuevas columnas con nombres consistentes.

## Preparar columnas antes de crear nuevas variables

Antes de crear columnas nuevas a partir de operaciones matemáticas, necesitamos asegurarnos de que las columnas involucradas sean realmente numéricas.

En nuestro dataset, las columnas `quantity`, `price_per_unit` y `total_spent` deberían representar cantidades o importes. Sin embargo, como vimos antes, el archivo original puede contener valores problemáticos como `"UNKNOWN"` o `"ERROR"`, y por eso Pandas puede haberlas cargado como texto.

Si intentamos multiplicar columnas que todavía están como texto, el resultado puede fallar o no representar un cálculo numérico real. Por eso, antes de crear una columna calculada, vamos a convertir esas variables a formato numérico.

In [21]:
columnas_numericas = [
    "quantity",
    "price_per_unit",
    "total_spent"
]

for columna in columnas_numericas:
    df_trabajo[columna] = pd.to_numeric(
        df_trabajo[columna],
        errors="coerce"
    )

df_trabajo[columnas_numericas].dtypes

,0
quantity,float64
price_per_unit,float64
total_spent,float64


Ahora las columnas seleccionadas fueron convertidas a tipo numérico.

Usamos `errors="coerce"` para que los valores que no puedan convertirse, como `"UNKNOWN"` o `"ERROR"`, pasen a ser `NaN`. Esto permite continuar con los cálculos, pero también nos obliga a revisar los faltantes resultantes.

Antes de crear nuevas columnas, conviene verificar cuántos valores faltantes tenemos en estas variables.

In [22]:
df_trabajo[columnas_numericas].isna().sum()

,0
quantity,479
price_per_unit,533
total_spent,502


Esta revisión nos recuerda que convertir tipos no elimina los problemas del dataset. Algunos valores quedan como faltantes porque no pudieron interpretarse como números.

Aun así, tener estas columnas en formato numérico nos permite crear nuevas variables calculadas y verificar relaciones internas.

## Crear una columna calculada

Una vez que `quantity` y `price_per_unit` están en formato numérico, podemos crear una nueva columna a partir de ellas.

En un dataset de ventas, una relación esperada es:

```text
importe calculado = cantidad × precio unitario
```

En nuestro caso, vamos a crear una columna llamada `calculated_total`.

Esta columna representa el total que se obtiene al multiplicar la cantidad vendida por el precio unitario.

In [23]:
df_trabajo["calculated_total"] = (
    df_trabajo["quantity"] * df_trabajo["price_per_unit"]
)

df_trabajo[
    [
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total"
    ]
].head(10)

,quantity,price_per_unit,total_spent,calculated_total
0,2.0,2.0,4.0,4.0
1,4.0,3.0,12.0,12.0
2,4.0,1.0,NaN,4.0
3,2.0,5.0,10.0,10.0
4,2.0,2.0,4.0,4.0
5,5.0,4.0,20.0,20.0
6,3.0,3.0,9.0,9.0
7,4.0,4.0,16.0,16.0
8,5.0,3.0,15.0,15.0
9,5.0,4.0,20.0,20.0


La nueva columna `calculated_total` no estaba en el dataset original. La construimos a partir de columnas existentes.

Este tipo de columna puede servir para enriquecer el análisis, pero también para validar datos. En este caso, el dataset ya tenía una columna llamada `total_spent`, que representa el total registrado de la transacción.

Entonces podemos comparar dos valores:

```text
total_spent        → total registrado en el archivo
calculated_total   → total calculado a partir de cantidad y precio unitario
```

Si ambos valores coinciden, la transacción parece consistente desde el punto de vista del importe. Si no coinciden, puede haber un error en alguna de las columnas o un problema de carga.

Esta es una idea muy importante: crear columnas nuevas no solo permite agregar información, también puede ayudar a controlar la calidad del dataset.


## Crear una columna de validación

La columna `calculated_total` nos permite comparar el importe registrado con el importe calculado.

Para eso podemos crear una nueva columna booleana llamada `total_matches`.

Esta columna indicará si `total_spent` coincide con `calculated_total`.

In [24]:
df_trabajo["total_matches"] = (
    df_trabajo["total_spent"] == df_trabajo["calculated_total"]
)

df_trabajo[
    [
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total",
        "total_matches"
    ]
].head(10)

,quantity,price_per_unit,total_spent,calculated_total,total_matches
0,2.0,2.0,4.0,4.0,True
1,4.0,3.0,12.0,12.0,True
2,4.0,1.0,NaN,4.0,False
3,2.0,5.0,10.0,10.0,True
4,2.0,2.0,4.0,4.0,True
5,5.0,4.0,20.0,20.0,True
6,3.0,3.0,9.0,9.0,True
7,4.0,4.0,16.0,16.0,True
8,5.0,3.0,15.0,15.0,True
9,5.0,4.0,20.0,20.0,True


La columna `total_matches` contiene valores `True` o `False`.

`True` significa que el total registrado coincide con el total calculado. `False` significa que no coinciden.

Sin embargo, debemos interpretar esta columna con cuidado. Si alguna de las columnas necesarias para el cálculo tiene un valor faltante, el resultado de la comparación puede ser `False`, aunque no necesariamente exista una inconsistencia real. Puede ocurrir simplemente que no tengamos suficiente información para validar esa fila.

Por eso, muchas veces conviene distinguir entre tres situaciones:

```text
el total coincide
el total no coincide
no hay datos suficientes para comparar
```

Vamos a crear una versión más expresiva de esta validación en la siguiente sección.

## Distinguir coincidencias, diferencias y datos insuficientes

La columna `total_matches` nos da una primera validación, pero es demasiado simple.

Cuando devuelve `False`, pueden estar ocurriendo dos cosas distintas. Tal vez `total_spent` y `calculated_total` realmente no coinciden. Pero también puede pasar que falte alguno de los datos necesarios para hacer la comparación.

Por ejemplo, si falta `quantity`, `price_per_unit` o `total_spent`, no tenemos información suficiente para validar esa fila.

Para evitar confusiones, podemos crear una columna más descriptiva llamada `estado_total`.

In [25]:
df_trabajo["estado_total"] = "sin_datos_suficientes"

condicion_datos_completos = (
    df_trabajo["quantity"].notna()
    & df_trabajo["price_per_unit"].notna()
    & df_trabajo["total_spent"].notna()
)

condicion_coincide = (
    condicion_datos_completos
    & (df_trabajo["total_spent"] == df_trabajo["calculated_total"])
)

condicion_no_coincide = (
    condicion_datos_completos
    & (df_trabajo["total_spent"] != df_trabajo["calculated_total"])
)

df_trabajo.loc[condicion_coincide, "estado_total"] = "coincide"
df_trabajo.loc[condicion_no_coincide, "estado_total"] = "no_coincide"

df_trabajo[
    [
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total",
        "estado_total"
    ]
].head(15)

,quantity,price_per_unit,total_spent,calculated_total,estado_total
0,2.0,2.0,4.0,4.0,coincide
1,4.0,3.0,12.0,12.0,coincide
2,4.0,1.0,NaN,4.0,sin_datos_suficientes
3,2.0,5.0,10.0,10.0,coincide
4,2.0,2.0,4.0,4.0,coincide
5,5.0,4.0,20.0,20.0,coincide
6,3.0,3.0,9.0,9.0,coincide
7,4.0,4.0,16.0,16.0,coincide
8,5.0,3.0,15.0,15.0,coincide
9,5.0,4.0,20.0,20.0,coincide


Ahora la validación es más clara.

La columna `estado_total` puede tomar tres valores:

```text
coincide
no_coincide
sin_datos_suficientes
```

Esto evita interpretar como error una fila que simplemente no tiene datos suficientes para ser evaluada.

Podemos revisar cuántos casos hay en cada grupo.

In [26]:
df_trabajo["estado_total"].value_counts(dropna=False)

,count
estado_total,
coincide,8544
sin_datos_suficientes,1456


Este conteo nos permite evaluar la consistencia interna del dataset. En la salida aparecen casos `coincide` y `sin_datos_suficientes`, pero no aparecen casos `no_coincide`. Eso indica que, dentro de las filas que tienen los datos necesarios para hacer la comparación, el total registrado coincide con el total calculado.

Las filas clasificadas como `sin_datos_suficientes` no indican una diferencia entre importes, sino falta de información para poder validar la relación.

Si hubiese muchas filas en `no_coincide`, deberíamos revisar si existe un problema en `quantity`, `price_per_unit`, `total_spent` o en la forma en que fue registrada la transacción.

Si hay muchas filas en `sin_datos_suficientes`, el problema principal no es una diferencia entre importes, sino la falta de información necesaria para validar.

Este tipo de columna es muy útil porque convierte una comparación técnica en una señal más interpretable para el análisis.

## Crear columnas a partir de una fecha

Además de crear columnas numéricas y columnas de validación, también podemos crear nuevas columnas a partir de una fecha.

En nuestro dataset, la columna `transaction_date` todavía necesita convertirse a formato temporal. Una vez convertida, podremos extraer información como el año, el mes o el día de la semana.

Primero convertimos la columna:

In [27]:
df_trabajo["transaction_date"] = pd.to_datetime(
    df_trabajo["transaction_date"],
    errors="coerce"
)

df_trabajo["transaction_date"].dtype

dtype('<M8[ns]')

Ahora `transaction_date` está en formato temporal.

Podemos crear algunas columnas derivadas:

In [28]:
df_trabajo["year"] = df_trabajo["transaction_date"].dt.year
df_trabajo["month"] = df_trabajo["transaction_date"].dt.month
df_trabajo["day_of_week"] = df_trabajo["transaction_date"].dt.day_name()

df_trabajo[
    [
        "transaction_date",
        "year",
        "month",
        "day_of_week"
    ]
].head(10)

,transaction_date,year,month,day_of_week
0,2023-09-08,2023.0,9.0,Friday
1,2023-05-16,2023.0,5.0,Tuesday
2,2023-07-19,2023.0,7.0,Wednesday
3,2023-04-27,2023.0,4.0,Thursday
4,2023-06-11,2023.0,6.0,Sunday
5,2023-03-31,2023.0,3.0,Friday
6,2023-10-06,2023.0,10.0,Friday
7,2023-10-28,2023.0,10.0,Saturday
8,2023-07-28,2023.0,7.0,Friday
9,2023-12-31,2023.0,12.0,Sunday


Estas columnas derivadas permiten analizar las transacciones desde una perspectiva temporal.

La columna `year` indica el año de la transacción. La columna `month` indica el número de mes. La columna `day_of_week` indica el día de la semana.

Observemos que `year` y `month` pueden aparecer con decimales, como `2023.0` o `9.0`. Esto ocurre porque algunas fechas no pudieron convertirse y quedaron como `NaT`. Al derivar columnas desde fechas faltantes, Pandas necesita usar un formato que también pueda representar esos valores ausentes.

También vemos que los nombres de los días aparecen en inglés, como `Friday`, `Tuesday` o `Wednesday`. Esto depende del entorno y no afecta el análisis: los valores siguen representando correctamente el día de la semana. Si más adelante necesitáramos presentar esos nombres en español, podríamos aplicar un reemplazo o mapeo de valores.

Al igual que con las columnas numéricas, estas nuevas variables no estaban explícitas en el dataset original. Las construimos a partir de una columna existente.

Crear columnas temporales puede ser útil para analizar períodos, detectar patrones por mes o estudiar diferencias entre días de la semana.

Si algunas filas tienen fechas faltantes o no interpretables, las columnas derivadas también quedarán con valores faltantes. Por eso, cada nueva columna debe interpretarse teniendo en cuenta la calidad de la columna original.

## Verificar las nuevas columnas

Después de renombrar columnas, convertir tipos y crear nuevas variables, conviene revisar cómo quedó el `DataFrame`.

La verificación nos ayuda a confirmar que las transformaciones se aplicaron correctamente y que las nuevas columnas tienen sentido.

Podemos comenzar revisando los nombres de columnas actuales.

In [29]:
df_trabajo.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date', 'calculated_total',
       'total_matches', 'estado_total', 'year', 'month', 'day_of_week'],
      dtype='object')

Ahora el `DataFrame` contiene tanto columnas originales renombradas como columnas nuevas creadas durante el proceso.

Entre las nuevas columnas tenemos:

```text
calculated_total
total_matches
estado_total
year
month
day_of_week
```

También podemos revisar una vista general de algunas columnas importantes.

In [30]:
df_trabajo[
    [
        "transaction_id",
        "item",
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total",
        "estado_total",
        "transaction_date",
        "year",
        "month",
        "day_of_week"
    ]
].head(10)

,transaction_id,item,quantity,price_per_unit,total_spent,calculated_total,estado_total,transaction_date,year,month,day_of_week
0,TXN_1961373,Coffee,2.0,2.0,4.0,4.0,coincide,2023-09-08,2023.0,9.0,Friday
1,TXN_4977031,Cake,4.0,3.0,12.0,12.0,coincide,2023-05-16,2023.0,5.0,Tuesday
2,TXN_4271903,Cookie,4.0,1.0,NaN,4.0,sin_datos_suficientes,2023-07-19,2023.0,7.0,Wednesday
3,TXN_7034554,Salad,2.0,5.0,10.0,10.0,coincide,2023-04-27,2023.0,4.0,Thursday
4,TXN_3160411,Coffee,2.0,2.0,4.0,4.0,coincide,2023-06-11,2023.0,6.0,Sunday
5,TXN_2602893,Smoothie,5.0,4.0,20.0,20.0,coincide,2023-03-31,2023.0,3.0,Friday
6,TXN_4433211,UNKNOWN,3.0,3.0,9.0,9.0,coincide,2023-10-06,2023.0,10.0,Friday
7,TXN_6699534,Sandwich,4.0,4.0,16.0,16.0,coincide,2023-10-28,2023.0,10.0,Saturday
8,TXN_4717867,NaN,5.0,3.0,15.0,15.0,coincide,2023-07-28,2023.0,7.0,Friday
9,TXN_2064365,Sandwich,5.0,4.0,20.0,20.0,coincide,2023-12-31,2023.0,12.0,Sunday


Esta vista permite observar, en una misma tabla, las variables originales ya preparadas y las nuevas columnas derivadas.

También podemos revisar los tipos de datos:

In [31]:
df_trabajo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              9667 non-null   object        
 2   quantity          9521 non-null   float64       
 3   price_per_unit    9467 non-null   float64       
 4   total_spent       9498 non-null   float64       
 5   payment_method    7421 non-null   object        
 6   location          6735 non-null   object        
 7   transaction_date  9540 non-null   datetime64[ns]
 8   calculated_total  9006 non-null   float64       
 9   total_matches     10000 non-null  bool          
 10  estado_total      10000 non-null  object        
 11  year              9540 non-null   float64       
 12  month             9540 non-null   float64       
 13  day_of_week       9540 non-null   object        
dtypes: bool(1), datetime64[

Esta revisión final es importante porque el `DataFrame` cambió bastante desde la carga inicial.

Primero estandarizamos nombres de columnas. Luego convertimos columnas numéricas. Después creamos columnas calculadas y columnas de validación. Finalmente convertimos la fecha y generamos columnas temporales.

Cada una de esas transformaciones modifica la estructura del dataset. Por eso, después de crear nuevas columnas, conviene revisar tanto los nombres como los tipos de datos y algunos ejemplos concretos.

Preparar datos no es solo transformar. También es verificar que cada transformación haya dejado el dataset en mejores condiciones para el análisis.

## Errores frecuentes al estandarizar nombres y crear columnas

Al estandarizar nombres de columnas, uno de los errores más frecuentes es cambiar nombres sin verificar después la estructura del `DataFrame`.

Si renombramos una columna y luego intentamos usar el nombre anterior, Pandas no va a encontrarla. Por eso, después de aplicar `rename()`, conviene revisar `df.columns` y asegurarnos de usar los nombres nuevos en las celdas siguientes.

Otro error posible es crear nuevas columnas antes de preparar los tipos de datos. Por ejemplo, si `quantity` o `price_per_unit` todavía estuvieran cargadas como texto, la columna `calculated_total` no sería confiable. Antes de crear variables numéricas derivadas, debemos convertir las columnas necesarias y revisar si aparecieron nuevos faltantes.

También debemos tener cuidado con las columnas de validación. Una comparación como:

```python
df_trabajo["total_spent"] == df_trabajo["calculated_total"]
```

puede devolver `False` cuando falta algún dato necesario para comparar. Eso no siempre significa que haya una inconsistencia real. Por eso creamos `estado_total`, que distingue entre coincidencia, diferencia y falta de datos suficientes.

Otro error frecuente es crear muchas columnas nuevas sin un propósito claro. Cada nueva variable debería tener una función: enriquecer el análisis, simplificar una consulta, validar otra columna o preparar el dataset para una etapa posterior.

Finalmente, al crear columnas derivadas de fechas, debemos recordar que dependen de la calidad de la fecha original. Si `transaction_date` no pudo convertirse correctamente y quedó como `NaT`, las columnas derivadas como `year`, `month` o `day_of_week` también quedarán sin información válida.

Una buena rutina para esta etapa podría ser:

```text
renombrar columnas con un criterio claro
verificar los nombres nuevos
convertir columnas necesarias antes de calcular
crear columnas derivadas con un propósito definido
crear columnas de validación cuando haya relaciones internas
verificar tipos, faltantes y ejemplos concretos
```

Estandarizar nombres y crear columnas nuevas no son pasos aislados. Forman parte de un proceso de preparación que busca dejar el dataset más claro, más consistente y más útil para el análisis.

## Resumen del capítulo

En este capítulo trabajamos con dos tareas importantes dentro de la preparación de datos: estandarizar nombres de columnas y crear nuevas columnas útiles para el análisis.

Primero retomamos la importancia de los nombres de columnas. El dataset original tenía nombres como `Transaction ID`, `Price Per Unit`, `Total Spent`, `Payment Method` y `Transaction Date`. Esos nombres son comprensibles, pero tienen espacios y mayúsculas, lo que puede volver el código menos cómodo de escribir.

Por eso creamos una copia del dataset y renombramos las columnas con un criterio uniforme:

```python
df_trabajo = df_trabajo.rename(columns={
    "Transaction ID": "transaction_id",
    "Item": "item",
    "Quantity": "quantity",
    "Price Per Unit": "price_per_unit",
    "Total Spent": "total_spent",
    "Payment Method": "payment_method",
    "Location": "location",
    "Transaction Date": "transaction_date"
})
```

El criterio elegido fue usar nombres en minúsculas, sin espacios y con guiones bajos. Este estilo facilita la escritura del código y ayuda a mantener una estructura más consistente.

Después preparamos algunas columnas antes de crear nuevas variables. Convertimos `quantity`, `price_per_unit` y `total_spent` a formato numérico usando `pd.to_numeric()` con `errors="coerce"`:

```python
for columna in columnas_numericas:
    df_trabajo[columna] = pd.to_numeric(
        df_trabajo[columna],
        errors="coerce"
    )
```

Este paso fue necesario porque no conviene crear columnas calculadas a partir de valores que todavía están cargados como texto.

Luego creamos una nueva columna llamada `calculated_total`:

```python
df_trabajo["calculated_total"] = (
    df_trabajo["quantity"] * df_trabajo["price_per_unit"]
)
```

Esta columna representa el total calculado a partir de la cantidad y el precio unitario.

Como el dataset ya tenía una columna `total_spent`, pudimos comparar el total registrado con el total calculado. Primero creamos una validación simple llamada `total_matches`, pero luego vimos que esa comparación podía ser insuficiente cuando faltaban datos.

Por eso creamos una columna más expresiva, `estado_total`, con tres estados posibles:

```text
coincide
no_coincide
sin_datos_suficientes
```

Esta columna permitió distinguir entre una inconsistencia real y una fila que no tenía datos suficientes para ser evaluada.

También convertimos `transaction_date` a formato temporal y creamos columnas derivadas de la fecha:

```python
df_trabajo["year"] = df_trabajo["transaction_date"].dt.year
df_trabajo["month"] = df_trabajo["transaction_date"].dt.month
df_trabajo["day_of_week"] = df_trabajo["transaction_date"].dt.day_name()
```

Estas columnas permiten analizar las transacciones desde una perspectiva temporal.

Finalmente verificamos las columnas creadas, los tipos de datos y algunos ejemplos concretos del `DataFrame` transformado.

La idea principal de este capítulo fue:

```text
Estandarizar nombres y crear columnas nuevas ayuda a transformar un dataset crudo en una tabla más clara, verificable y útil para el análisis.
```

Crear columnas no es solo agregar información. También puede servir para validar relaciones internas, detectar inconsistencias y preparar el dataset para análisis posteriores.


## Próximo paso

Ya trabajamos por separado varias tareas de preparación: valores faltantes, duplicados, limpieza de textos, conversión de tipos, fechas, estandarización de nombres y creación de nuevas columnas.

El siguiente paso será integrar varias de estas tareas en un flujo de limpieza encadenado.

En lugar de trabajar cada problema por separado, vamos a partir de un dataset crudo y construir una versión preparada paso a paso. La idea será aplicar transformaciones en un orden razonable, verificar resultados intermedios y obtener un `DataFrame` final más limpio y listo para analizar.